# HR-only CNN sleep-stage classifier

This notebook is the comparison baseline for the multimodal CNN (`sleep_classifier_cnn.ipynb`). The network is identical to the HR branch of the multimodal notebook (`EpochContextBranch` with one input channel plus a linear head, i.e. stage 1 of the staged model), and the data pipeline, context windows, 2 Hz input rate, 10-second training stride, class-weighted label-smoothed loss, AdamW optimizer, learning-rate schedule, early stopping, labels, and metrics are all the same. The only difference is that the CIN2/CIN4 eye-sensor channels are not used. The FDC timestamps are still read so every sleep is trimmed to the same common duration, which makes both notebooks score exactly the same validation epochs.

By default one complete sleep (`VALIDATION_DATE`) is held out for validation. With `CROSS_VALIDATE = True` the model is trained once per sleep with that sleep held out, each fold is reported, and the mean and standard deviation over the folds are printed at the end.

In [7]:
from __future__ import annotations

import copy
import math
import re
from pathlib import Path

import numpy as np
from IPython.display import display
import pandas as pd
import torch
from scipy.signal import resample_poly
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

DATASET_DIR = Path.cwd()
if DATASET_DIR.name != "datasets_for_training":
    raise RuntimeError("Open and run this notebook with datasets_for_training as the working directory.")
VALIDATION_DATE = "2026-10-17"
CROSS_VALIDATE = True  # True: train once per sleep with that sleep held out, then print the average over all folds
SAMPLE_RATE = 20
MODEL_RATE = 2
EPOCH_SECONDS = 30
TRAIN_STRIDE_SECONDS = 10
CONTEXT_BEFORE = 5
CONTEXT_AFTER = 4
CONTEXT_EPOCHS = CONTEXT_BEFORE + 1 + CONTEXT_AFTER
SAMPLES_PER_EPOCH = EPOCH_SECONDS * MODEL_RATE
INPUT_SAMPLES = CONTEXT_EPOCHS * SAMPLES_PER_EPOCH
BATCH_SIZE = 100
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-2
DROPOUT = 0.3
ENCODER_WIDTH = 32
LABEL_SMOOTHING = 0.1
CLASS_WEIGHT_POWER = 0.5
MAX_EPOCHS = 300
LR_PATIENCE = 10
EARLY_STOPPING_PATIENCE = 30
STAGES = ["Awake", "REM", "Core", "Deep"]
STAGE_TO_INDEX = {stage: index for index, stage in enumerate(STAGES)}
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is required by this notebook, but torch.cuda.is_available() is False.")
DEVICE = torch.device("cuda")
# GPU speed-ups: cuDNN picks the fastest convolution algorithms, and forward/backward passes run in bfloat16
# mixed precision (BatchNorm and the loss stay in float32). Set AUTOCAST_DTYPE = None for full float32.
torch.backends.cudnn.benchmark = True
AUTOCAST_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else None
print("Dataset:", DATASET_DIR)
print("Device:", torch.cuda.get_device_name(DEVICE))
print("PyTorch:", torch.__version__, "| mixed precision:", AUTOCAST_DTYPE)
print("Validation:", "every sleep in turn (cross-validation)" if CROSS_VALIDATE else VALIDATION_DATE)

Dataset: c:\BME\Szemfedo\CapMeter\data_analysis\sleep_tests\datasets_for_training
Device: NVIDIA GeForce RTX 3050 6GB Laptop GPU
PyTorch: 2.11.0+cu128 | mixed precision: torch.bfloat16
Validation: every sleep in turn (cross-validation)


## Loading and aligning the training files

Apple Watch stages are labels; the Polar heart rate is the sole input signal. Exactly as in the multimodal notebook, the 1 Hz heart rate is linearly interpolated onto a 20 Hz grid covering the interval common to the Apple Watch, Polar and FDC recordings. Only the FDC timestamps are read (for that common duration); the capacitive channels are not used.

In [8]:
def _date_from_name(path: Path) -> str:
    match = re.fullmatch(r"(\d{4}-\d{2}-\d{2})", path.stem)
    if match is None:
        raise ValueError(f"Unexpected Apple Watch filename: {path.name}")
    return match.group(1)


def _find_files(recording_date: str) -> tuple[Path, Path, Path]:
    apple = DATASET_DIR / f"{recording_date}.csv"
    fdc = DATASET_DIR / f"fdc1004_data_{recording_date.replace('-', '_')}.csv"
    polar_files = sorted(DATASET_DIR.glob(f"polar_{recording_date}_*.CSV"))
    if not apple.is_file() or not fdc.is_file() or not polar_files:
        raise FileNotFoundError(f"Missing one or more files for {recording_date}")
    return apple, polar_files[0], fdc


def _load_sleep(recording_date: str) -> pd.DataFrame:
    apple_path, polar_path, fdc_path = _find_files(recording_date)
    apple = pd.read_csv(apple_path)
    for column in ("Start time", "End time"):
        apple[column] = pd.to_datetime(apple[column]).map(
            lambda value: value.tz_localize(None) if value.tzinfo else value
        )
    apple_start = apple["Start time"].min()
    apple["start_s"] = (apple["Start time"] - apple_start).dt.total_seconds()
    apple["end_s"] = (apple["End time"] - apple_start).dt.total_seconds()

    polar = pd.read_csv(polar_path)
    polar["time_s"] = pd.to_timedelta(polar["Time"]).dt.total_seconds()
    polar["time_s"] -= polar["time_s"].min()
    polar["HR (bpm)"] = pd.to_numeric(polar["HR (bpm)"], errors="coerce")
    polar = polar.dropna(subset=["time_s", "HR (bpm)"]).drop_duplicates("time_s")

    # Same row filtering as the multimodal notebook, so the common duration matches exactly.
    fdc = pd.read_csv(fdc_path, usecols=["timestamp_ms", "CIN2_pF", "CIN4_pF"], low_memory=False)
    timestamp_ms = pd.to_numeric(fdc["timestamp_ms"], errors="coerce")
    fdc["time_s"] = (timestamp_ms - timestamp_ms.iloc[0]) / 1000
    for column in ("CIN2_pF", "CIN4_pF"):
        fdc[column] = pd.to_numeric(fdc[column], errors="coerce")
    fdc = fdc.dropna(subset=["time_s", "CIN2_pF", "CIN4_pF"])

    duration = min(float(apple["end_s"].max()), float(polar["time_s"].max()), float(fdc["time_s"].max()))
    samples = np.arange(0, math.floor(duration * SAMPLE_RATE) + 1, dtype=np.float32)
    seconds = samples / SAMPLE_RATE
    result = pd.DataFrame({"time_s": seconds})
    result["HR (bpm)"] = np.interp(seconds, polar["time_s"], polar["HR (bpm)"])
    result["stage"] = pd.NA
    for row in apple.itertuples():
        mask = (result["time_s"] >= row.start_s) & (result["time_s"] < row.end_s)
        result.loc[mask, "stage"] = row._1
    return result.dropna(subset=["stage"]).reset_index(drop=True)


recording_dates = sorted(_date_from_name(path) for path in DATASET_DIR.glob("20??-??-??.csv"))
if len(recording_dates) != 4:
    raise ValueError(f"Expected four training sleeps, found: {recording_dates}")
if VALIDATION_DATE not in recording_dates:
    raise ValueError(f"VALIDATION_DATE={VALIDATION_DATE} is not one of {recording_dates}")
sleep_data = {recording_date: _load_sleep(recording_date) for recording_date in recording_dates}
for recording_date, data in sleep_data.items():
    print(recording_date, len(data), "20-Hz rows", data["stage"].value_counts().to_dict())

2026-10-16 428881 20-Hz rows {'Core': 228381, 'REM': 107460, 'Deep': 78620, 'Awake': 14420}
2026-10-17 421861 20-Hz rows {'Core': 177621, 'Deep': 108020, 'REM': 103800, 'Awake': 32420}
2026-10-19 397347 20-Hz rows {'Core': 196240, 'Deep': 93020, 'REM': 89487, 'Awake': 18600}
2026-10-20 176700 20-Hz rows {'Core': 105100, 'Deep': 52820, 'REM': 18780}


## Context windows and labels

Identical to the multimodal notebook: the 20 Hz signal is decimated to `MODEL_RATE` with an anti-aliasing polyphase filter, each label is the majority Apple Watch stage in the center 30-second epoch, and each input holds 5 preceding epochs, the target, and 4 following epochs. Training windows start every `TRAIN_STRIDE_SECONDS`; validation windows are non-overlapping.

In [9]:
def _make_examples(data: pd.DataFrame, stride_seconds: int = EPOCH_SECONDS) -> tuple[np.ndarray, np.ndarray]:
    factor = SAMPLE_RATE // MODEL_RATE
    usable_samples = len(data) // factor * factor
    values = data[["HR (bpm)"]].to_numpy(np.float64)[:usable_samples]
    values = resample_poly(values, 1, factor, axis=0).astype(np.float32)
    labels = data["stage"].map(STAGE_TO_INDEX).to_numpy()[:usable_samples:factor][:len(values)]
    window = CONTEXT_EPOCHS * SAMPLES_PER_EPOCH
    examples, targets = [], []
    for start in range(0, len(values) - window + 1, stride_seconds * MODEL_RATE):
        center = start + CONTEXT_BEFORE * SAMPLES_PER_EPOCH
        examples.append(values[start:start + window])
        targets.append(int(np.bincount(labels[center:center + SAMPLES_PER_EPOCH], minlength=len(STAGES)).argmax()))
    return np.asarray(examples, dtype=np.float32), np.asarray(targets, dtype=np.int64)


def _standardize(train_x: np.ndarray, other_x: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    mean = train_x.reshape(-1, train_x.shape[-1]).mean(axis=0)
    std = train_x.reshape(-1, train_x.shape[-1]).std(axis=0)
    std[std < 1e-6] = 1.0
    return (train_x - mean) / std, (other_x - mean) / std


def _prepare_fold(validation_date: str) -> tuple[DataLoader, DataLoader, torch.Tensor]:
    """Train on every other sleep, validate on `validation_date`."""
    train_parts = [_make_examples(sleep_data[d], TRAIN_STRIDE_SECONDS) for d in recording_dates if d != validation_date]
    validation_x, validation_y = _make_examples(sleep_data[validation_date])
    train_x = np.concatenate([part[0] for part in train_parts])
    train_y = np.concatenate([part[1] for part in train_parts])
    train_x, validation_x = _standardize(train_x, validation_x)
    train_x = torch.from_numpy(train_x.transpose(0, 2, 1))
    validation_x = torch.from_numpy(validation_x.transpose(0, 2, 1))
    train_y = torch.from_numpy(train_y)
    validation_y = torch.from_numpy(validation_y)
    print("Train:", tuple(train_x.shape), "| Validation:", tuple(validation_x.shape))
    train_loader = DataLoader(TensorDataset(train_x, train_y), batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)
    validation_loader = DataLoader(TensorDataset(validation_x, validation_y), batch_size=BATCH_SIZE, shuffle=False, pin_memory=True)
    return train_loader, validation_loader, train_y

In [10]:
class FastMaxPool1d(nn.Module):
    """Same result as nn.MaxPool1d(2) (an odd last sample is dropped), but much faster backward pass on the GPU."""

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch, channels, length = x.shape
        return x[..., : length // 2 * 2].reshape(batch, channels, length // 2, 2).amax(-1)


def _autocast() -> torch.autocast:
    return torch.autocast("cuda", dtype=AUTOCAST_DTYPE or torch.float32, enabled=AUTOCAST_DTYPE is not None)


class EpochContextBranch(nn.Module):
    """Shared per-epoch encoder followed by a convolution across the 10 context epochs.
    Identical to the branch used in sleep_classifier_cnn.ipynb."""

    def __init__(self, channels: int, width: int = ENCODER_WIDTH) -> None:
        super().__init__()
        blocks = []
        in_channels = channels
        for out_channels in (width, width, 2 * width, 2 * width):
            blocks.extend([
                nn.Conv1d(in_channels, out_channels, kernel_size=7, padding=3, bias=False),
                nn.BatchNorm1d(out_channels),
                nn.ReLU(),
                FastMaxPool1d(),
            ])
            in_channels = out_channels
        # Shared encoder: one 30-second epoch -> one embedding vector.
        self.epoch_encoder = nn.Sequential(*blocks, nn.AdaptiveAvgPool1d(1), nn.Flatten(), nn.Dropout(DROPOUT))
        # Mixes neighbouring epoch embeddings across the 10-epoch context.
        self.context = nn.Sequential(
            nn.Conv1d(in_channels, in_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm1d(in_channels),
            nn.ReLU(),
            nn.Dropout(DROPOUT),
        )
        self.out_features = 2 * in_channels

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch, channels, _ = x.shape
        epochs = x.view(batch, channels, CONTEXT_EPOCHS, SAMPLES_PER_EPOCH).permute(0, 2, 1, 3)
        embeddings = self.epoch_encoder(epochs.reshape(batch * CONTEXT_EPOCHS, channels, SAMPLES_PER_EPOCH))
        context = self.context(embeddings.view(batch, CONTEXT_EPOCHS, -1).transpose(1, 2))
        return torch.cat([context[:, :, CONTEXT_BEFORE], context.mean(2)], 1)


def _initialize(module: nn.Module) -> None:
    if isinstance(module, (nn.Conv1d, nn.Linear)):
        nn.init.kaiming_normal_(module.weight, mode="fan_in", nonlinearity="relu")
        if module.bias is not None:
            nn.init.zeros_(module.bias)


class HRSleepCNN(nn.Module):
    """Same layers as the HR part of StagedSleepCNN (`hr` branch + `hr_head`) in the multimodal notebook."""

    def __init__(self, classes: int = 4) -> None:
        super().__init__()
        self.hr = EpochContextBranch(1)
        self.hr_head = nn.Linear(self.hr.out_features, classes)
        self.apply(_initialize)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.hr_head(self.hr(x))


def _build_model() -> nn.Module:
    return HRSleepCNN().to(DEVICE)


def _train_loss_function(train_y: torch.Tensor) -> nn.Module:
    class_counts = torch.bincount(train_y, minlength=len(STAGES)).float().clamp(min=1)
    class_weights = (class_counts.sum() / class_counts) ** CLASS_WEIGHT_POWER
    class_weights = (class_weights / class_weights.mean()).to(DEVICE)
    return nn.CrossEntropyLoss(weight=class_weights, label_smoothing=LABEL_SMOOTHING)


loss_function = nn.CrossEntropyLoss()  # unweighted, used for validation / early stopping
example_model = _build_model()
print(example_model)
print("Parameters:", sum(p.numel() for p in example_model.parameters()))

HRSleepCNN(
  (hr): EpochContextBranch(
    (epoch_encoder): Sequential(
      (0): Conv1d(1, 32, kernel_size=(7,), stride=(1,), padding=(3,), bias=False)
      (1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU()
      (3): FastMaxPool1d()
      (4): Conv1d(32, 32, kernel_size=(7,), stride=(1,), padding=(3,), bias=False)
      (5): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (6): ReLU()
      (7): FastMaxPool1d()
      (8): Conv1d(32, 64, kernel_size=(7,), stride=(1,), padding=(3,), bias=False)
      (9): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (10): ReLU()
      (11): FastMaxPool1d()
      (12): Conv1d(64, 64, kernel_size=(7,), stride=(1,), padding=(3,), bias=False)
      (13): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (14): ReLU()
      (15): FastMaxPool1d()
      (16): AdaptiveAvgPool1d(output_size=1)
 

In [11]:
def _evaluate(model: nn.Module, loader: DataLoader) -> tuple[float, np.ndarray]:
    model.eval()
    total_loss, predictions, targets = 0.0, [], []
    with torch.no_grad():
        for features, labels in loader:
            features = features.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            with _autocast():
                logits = model(features)
            logits = logits.float()
            total_loss += loss_function(logits, labels).item() * labels.size(0)
            predictions.append(logits.argmax(1).cpu().numpy())
            targets.append(labels.cpu().numpy())
    return total_loss / len(loader.dataset), np.column_stack((np.concatenate(targets), np.concatenate(predictions)))


def _fit(
    model: nn.Module,
    train_loader: DataLoader,
    validation_loader: DataLoader,
    train_loss_function: nn.Module,
    parameter_groups,
    label: str,
) -> float:
    """Train until early stopping; same loop as the multimodal notebook."""
    optimizer = torch.optim.AdamW(parameter_groups, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=LR_PATIENCE)
    best_loss, _ = _evaluate(model, validation_loader)
    best_state, stale = copy.deepcopy(model.state_dict()), 0
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        for features, labels in train_loader:
            features = features.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with _autocast():
                loss = train_loss_function(model(features), labels)
            loss.backward()
            optimizer.step()
        validation_loss, _ = _evaluate(model, validation_loader)
        scheduler.step(validation_loss)
        if validation_loss < best_loss - 1e-6:
            best_loss, best_state, stale = validation_loss, copy.deepcopy(model.state_dict()), 0
        else:
            stale += 1
        if epoch == 1 or epoch % 10 == 0:
            print(f"[{label}] epoch={epoch:03d} validation_loss={validation_loss:.5f} lr={optimizer.param_groups[0]['lr']:.2e}")
        if stale >= EARLY_STOPPING_PATIENCE:
            print(f"[{label}] early stopping at epoch", epoch)
            break
    model.load_state_dict(best_state)
    print(f"[{label}] best validation loss:", best_loss)
    return best_loss


def _train(model: nn.Module, train_loader: DataLoader, validation_loader: DataLoader, train_y: torch.Tensor) -> None:
    _fit(model, train_loader, validation_loader, _train_loss_function(train_y), model.parameters(), "HR")

In [12]:
def classification_report(
    targets: np.ndarray, predictions: np.ndarray, validation_loss: float, validation_date: str
) -> tuple[pd.DataFrame, pd.Series]:
    matrix = np.zeros((len(STAGES), len(STAGES)), dtype=np.int64)
    for target, prediction in zip(targets, predictions):
        matrix[target, prediction] += 1
    rows = []
    for index, stage in enumerate(STAGES):
        tp = matrix[index, index]
        precision = tp / matrix[:, index].sum() if matrix[:, index].sum() else 0.0
        recall = tp / matrix[index].sum() if matrix[index].sum() else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        rows.append({"stage": stage, "precision": precision, "recall": recall, "f1": f1, "support": matrix[index].sum()})
    report = pd.DataFrame(rows).set_index("stage")
    # Stages absent from the validation night are excluded from macro averages.
    present = report[report["support"] > 0]
    macro = present[["precision", "recall", "f1"]].mean()
    weighted = present[["precision", "recall", "f1"]].mul(present["support"], axis=0).sum() / present["support"].sum()
    report.loc["macro avg"] = [*macro, present["support"].sum()]
    report.loc["weighted avg"] = [*weighted, present["support"].sum()]
    report["support"] = report["support"].astype(int)

    total = matrix.sum()
    accuracy = np.trace(matrix) / total if total else 0.0
    expected = (matrix.sum(axis=0) * matrix.sum(axis=1)).sum() / total**2 if total else 0.0
    kappa = (accuracy - expected) / (1 - expected) if expected < 1 else 0.0
    awake = STAGE_TO_INDEX["Awake"]
    true_sleep, predicted_sleep = targets != awake, predictions != awake
    sleep_wake_accuracy = (true_sleep == predicted_sleep).mean()
    tst_true = true_sleep.sum() * EPOCH_SECONDS / 3600
    tst_predicted = predicted_sleep.sum() * EPOCH_SECONDS / 3600
    summary = pd.Series({
        "Validation loss": validation_loss,
        "Accuracy": accuracy,
        "Balanced accuracy (macro recall)": macro["recall"],
        "Macro F1": macro["f1"],
        "Weighted F1": weighted["f1"],
        "Cohen kappa": kappa,
        "Sleep/wake accuracy": sleep_wake_accuracy,
        "Sleep sensitivity": predicted_sleep[true_sleep].mean() if true_sleep.any() else np.nan,
        "Wake specificity": (~predicted_sleep[~true_sleep]).mean() if (~true_sleep).any() else np.nan,
        "TST ground truth (h)": tst_true,
        "TST CNN (h)": tst_predicted,
        "TST error (min)": (tst_predicted - tst_true) * 60,
    })
    print("Validation sleep:", validation_date, "|", total, "epochs")
    display(summary.round(4).to_frame("value"))
    print("Confusion matrix (rows=true, columns=predicted):")
    display(pd.DataFrame(matrix, index=STAGES, columns=STAGES))
    print("Stage durations (minutes):")
    display(pd.DataFrame({
        "true": matrix.sum(axis=1) * EPOCH_SECONDS / 60,
        "predicted": matrix.sum(axis=0) * EPOCH_SECONDS / 60,
    }, index=STAGES))
    display(report.round(3))
    return report, summary


def run_fold(validation_date: str) -> tuple[nn.Module, pd.DataFrame, pd.Series]:
    torch.manual_seed(SEED)
    np.random.seed(SEED)
    print(f"\n========== Validation sleep: {validation_date} ==========")
    train_loader, validation_loader, train_y = _prepare_fold(validation_date)
    model = HRSleepCNN().to(DEVICE)
    _train(model, train_loader, validation_loader, train_y)
    validation_loss, result = _evaluate(model, validation_loader)
    report, summary = classification_report(result[:, 0], result[:, 1], validation_loss, validation_date)
    return model, report, summary


validation_dates = recording_dates if CROSS_VALIDATE else [VALIDATION_DATE]
fold_results = {validation_date: run_fold(validation_date) for validation_date in validation_dates}
model, report, summary = fold_results[validation_dates[-1]]  # last fold, kept for further inspection

if CROSS_VALIDATE:
    summaries = pd.DataFrame({date: fold[2] for date, fold in fold_results.items()}).T
    stage_f1 = pd.DataFrame({
        date: fold[1]["f1"].where(fold[1]["support"] > 0)  # absent stages -> NaN, skipped by mean
        for date, fold in fold_results.items()
    }).T
    for table in (summaries, stage_f1):
        mean, std = table.mean(), table.std()
        table.loc["mean"], table.loc["std"] = mean, std
    print(f"\n========== Leave-one-night-out summary ({len(validation_dates)} folds) ==========")
    display(summaries.round(3))
    print("Per-stage F1 by validation night:")
    display(stage_f1.round(3))


========== Validation sleep: 2026-10-16 ==========
Train: (4891, 1, 600) | Validation: (705, 1, 600)
[HR] epoch=001 validation_loss=1.69135 lr=1.00e-03
[HR] epoch=010 validation_loss=1.41297 lr=1.00e-03
[HR] epoch=020 validation_loss=1.42336 lr=5.00e-04
[HR] epoch=030 validation_loss=1.34695 lr=5.00e-04
[HR] epoch=040 validation_loss=1.23256 lr=2.50e-04
[HR] epoch=050 validation_loss=1.23031 lr=1.25e-04
[HR] early stopping at epoch 56
[HR] best validation loss: 1.1542370739980792
Validation sleep: 2026-10-16 | 705 epochs


,value
Validation loss,1.1542
Accuracy,0.5872
Balanced accuracy (macro recall),0.3907
Macro F1,0.3560
Weighted F1,0.5252
Cohen kappa,0.2808
Sleep/wake accuracy,0.9603
Sleep sensitivity,0.9927
Wake specificity,0.0417
TST ground truth (h),5.6750


Confusion matrix (rows=true, columns=predicted):


,Awake,REM,Core,Deep
Awake,1,6,17,0
REM,0,136,43,0
Core,5,89,274,3
Deep,0,10,118,3


Stage durations (minutes):


...

,precision,recall,f1,support
stage,,,,
Awake,0.000,0.000,0.000,0
REM,0.429,0.556,0.484,27
Core,0.564,0.547,0.555,170
Deep,0.342,0.284,0.311,88
macro avg,0.445,0.462,0.450,285
weighted avg,0.483,0.467,0.473,285



========== Leave-one-night-out summary (4 folds) ==========


,Validation loss,Accuracy,Balanced accuracy (macro recall),Macro F1,Weighted F1,Cohen kappa,Sleep/wake accuracy,Sleep sensitivity,Wake specificity,TST ground truth (h),TST CNN (h),TST error (min)
2026-10-16,1.154,0.587,0.391,0.356,0.525,0.281,0.960,0.993,0.042,5.675,5.825,9.000
2026-10-17,1.120,0.549,0.543,0.508,0.552,0.385,0.836,0.873,0.389,5.333,4.933,-24.000
2026-10-19,0.994,0.608,0.492,0.476,0.602,0.406,0.922,0.968,0.000,5.183,5.275,5.500
2026-10-20,1.279,0.467,0.462,0.450,0.473,0.054,0.958,0.958,NaN,2.375,2.275,-6.000
mean,1.137,0.553,0.472,0.448,0.538,0.282,0.919,0.948,0.144,4.642,4.577,-3.875
std,0.117,0.062,0.064,0.066,0.054,0.161,0.058,0.052,0.214,1.525,1.578,14.868


Per-stage F1 by validation night:


stage,Awake,REM,Core,Deep,macro avg,weighted avg
2026-10-16,0.067,0.648,0.666,0.044,0.356,0.525
2026-10-17,0.269,0.706,0.527,0.530,0.508,0.552
2026-10-19,0.000,0.677,0.628,0.599,0.476,0.602
2026-10-20,NaN,0.484,0.555,0.311,0.450,0.473
mean,0.112,0.629,0.594,0.371,0.448,0.538
std,0.140,0.100,0.064,0.250,0.066,0.054
